Implementing the Macro method from Couloumbe (2024)

Data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm

In [ ]:
df = pd.read_csv('england_master.csv')

In [ ]:
df['date'] = pd.to_datetime(df['Unnamed: 0'].str.replace('Q', '-Q'))
df.set_index('date', inplace=True)

In [ ]:
df_clean = df[['starts', 'hprice', 'cc', 'rate', 'vol', 'gdp_def']].copy()

In [ ]:
df_clean['starts_lag1'] = df_clean['starts'].shift(1)
df_clean['starts_lag4'] = df_clean['starts'].shift(4)
df_clean['vol_lag1'] = df_clean['vol'].shift(1)
df_clean['rate_lag1'] = df_clean['rate'].shift(1)
df_clean['time_trend'] = np.arange(len(df_clean))

In [ ]:
df_clean = df_clean.dropna()

In [ ]:
df_clean['starts'] = np.log(df_clean['starts'])
df_clean['hprice'] = np.log(df_clean['hprice'] / df_clean['gdp_def'])  # deflated by gdp_def, matching 04_ARRF.py
df_clean['cc'] = np.log(df_clean['cc'] / df_clean['gdp_def'])  # deflated by gdp_def, matching 04_ARRF.py

In [ ]:
y = df_clean['starts']  # log(starts) now that the log transform runs first, consistent with lhstarts convention

In [ ]:
# OPEN DESIGN QUESTION: S/X split is reversed vs Coulombe's AR special case (AR lags in forest state S here, price/cost/rate in linear block X) -- deliberate MRF-style variant, left as-is, not a strict ARRF replication
X = df_clean[['hprice', 'cc', 'rate']]
X = sm.add_constant(X)

In [ ]:
S = df_clean[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]
print(f"Data cleaned. Proceeding with {len(y)} perfectly aligned quarters.")

The macroeconomic RF

In [ ]:
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf.fit(S, y)

In [ ]:
leaf_assignments = rf.apply(S)

In [ ]:
valid_idx = df_clean.index
gtvps = pd.DataFrame(index=valid_idx, columns=X.columns, dtype=float)
predicted_y = pd.Series(index=valid_idx, dtype=float)

In [ ]:
# Freeze ridge lambda via leaf-weighted CV on the pre-2010Q1 subsample only: a flat unweighted TimeSeriesSplit fit picked lambda at the edge of the grid and zeroed every slope, so each candidate alpha is scored using the same leaf-co-occurrence weights as the main loop (restricted to each fold's training positions), closer to Friedberg et al.'s local-linear-forest tuning than a single global fit
X_no_const = X.drop(columns='const')
pre_idx = np.where(df_clean.index < pd.Timestamp('2010-01-01'))[0]
alphas = np.logspace(-3, 5, 17)
tscv = TimeSeriesSplit(n_splits=5)
cv_errors = {a: [] for a in alphas}
for train_pos, test_pos in tscv.split(pre_idx):
    train_idx = pre_idx[train_pos]
    for t in pre_idx[test_pos]:
        w = np.sum(leaf_assignments[train_idx] == leaf_assignments[t], axis=1)
        if w.sum() == 0:
            continue
        w = w / w.sum()
        for a in alphas:
            m = Ridge(alpha=a).fit(X_no_const.iloc[train_idx], y.iloc[train_idx], sample_weight=w)
            pred = m.predict(X_no_const.iloc[[t]])[0]
            cv_errors[a].append((y.iloc[t] - pred) ** 2)
lam = min(alphas, key=lambda a: np.mean(cv_errors[a]))
print(f"Selected ridge lambda (leaf-weighted CV, pre-2010Q1): {lam}")

In [ ]:
# Leaf-local ridge WLS (Friedberg et al. 2020 local linear forest), replacing the unregularized sm.WLS flagged above
for t_idx in range(len(valid_idx)):
  current_leaves = leaf_assignments[t_idx, :]
  weights = np.sum(leaf_assignments == current_leaves, axis=1)
  weights = weights / weights.sum()
  ridge_model = Ridge(alpha=lam).fit(X_no_const, y, sample_weight=weights)
  gtvps.iloc[t_idx] = np.concatenate([[ridge_model.intercept_], ridge_model.coef_])
  predicted_y.iloc[t_idx] = ridge_model.predict(X_no_const.iloc[[t_idx]])[0]

In [ ]:
gtvps = gtvps.astype(float)

Visualisation


In [ ]:
#Plot1
plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['rate'], color='crimson', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of the BoE Base Rate on Housing Starts', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Rate)', fontsize=12)
plt.grid(True, alpha=0.3)

plt.fill_between(gtvps.index, gtvps['rate'], 0, where=(gtvps['rate'] < 0), color='crimson', alpha=0.1)

plt.tight_layout()
plt.savefig('gtvp_interest_rate.png')
plt.show()

In [ ]:
#plot2

plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['cc'], color='darkblue', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of Construction Costs', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Costs)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gtvp_construction_costs.png')
plt.show()